# M1 · Tensors, autograd & PyTorch essentials

**No experience needed.** This lesson assumes you've never trained a neural network and never used PyTorch. You only need to be able to read a little Python.

By the end you'll understand, line by line, the code that trains a small neural network. Every fine-tune later in the course is that same code, just with a bigger model.

**How to work through it**

- Run each cell from top to bottom with **Shift-Enter**.
- After each code cell there's a **What each line does** box. Read it before moving on.
- When a cell says **Predict**, write your guess down *before* running it. Being wrong and seeing why is how this sticks.
- New words are in **bold** where they're explained. They're also in the glossary in `README.md`.
- Save with **Cmd-S** now and then, so the dashboard can see your progress.

## 0. What is a model, and what does training mean?

Forget neural networks for a moment. A **model** is just a function with some adjustable numbers inside it.

Say you want to predict a house's price from its size. A very simple model is

&nbsp;&nbsp;&nbsp;&nbsp;`price = w × size + b`

where `w` and `b` are the adjustable numbers. They're called **parameters**. `w` is a *weight* (it scales the input) and `b` is a **bias** (a number added on at the end). Pick good values and the predictions are close; pick bad ones and they're way off.

**Training** means finding good values for the parameters automatically, by repeating four steps:

1. **Guess.** Run the model on some examples with its current parameters.
2. **Measure.** Work out one number that says how wrong the guesses were. This number is the **loss**. Smaller is better.
3. **Find the direction.** For each parameter, work out whether nudging it up or down would make the loss smaller. That's what a *gradient* tells you (section 2).
4. **Nudge.** Move every parameter a small step in its better direction, then go back to step 1.

That's it. A large language model is the same idea with billions of parameters instead of two. The rest of this lesson is just the PyTorch tools that do each of those four steps.

Run the cell below to see the loop as a picture.

In [ ]:
import matplotlib.pyplot as plt

steps = [
    "1. Guess\n(run the model)",
    "2. Measure\n(compute the loss)",
    "3. Find the direction\n(gradients)",
    "4. Nudge\n(update parameters)",
]
xy = [(0.5, 0.85), (0.85, 0.5), (0.5, 0.15), (0.15, 0.5)]

fig, ax = plt.subplots(figsize=(5, 4))
for (x, y), label in zip(xy, steps, strict=True):
    ax.text(
        x,
        y,
        label,
        ha="center",
        va="center",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.5", fc="#e6ecfd", ec="#2e57e8"),
    )
for i in range(4):
    (x0, y0), (x1, y1) = xy[i], xy[(i + 1) % 4]
    ax.annotate(
        "",
        xy=(x0 + 0.6 * (x1 - x0), y0 + 0.6 * (y1 - y0)),
        xytext=(x0 + 0.4 * (x1 - x0), y0 + 0.4 * (y1 - y0)),
        arrowprops=dict(arrowstyle="->", lw=1.5),
    )
ax.set_title("The training loop: repeat until the loss stops falling")
ax.axis("off")
plt.show()

**What each line does**

- `import matplotlib.pyplot as plt`: loads the plotting library. `plt` is the short name everyone uses for it.
- `steps`, `xy`: the four labels and where to place each one on the picture.
- `plt.subplots(...)`: makes an empty picture (`fig`) with one drawing area (`ax`).
- `ax.text(...)`: writes each label inside a rounded box. `ax.annotate(...)` draws the arrows between them.
- `plt.show()`: displays the picture. None of this is PyTorch. It's here only so you can see the loop.

## 1. Tensors

A **tensor** is a grid of numbers. That's all. It's how PyTorch stores everything: the inputs, the parameters, the predictions and the gradients.

- A single number is a 0-dimensional tensor (like `7.0`).
- A list of numbers is 1-dimensional (like `[1.0, 2.0, 3.0]`).
- A table of numbers is 2-dimensional (rows and columns).
- Tensors can have more dimensions too, but 0, 1 and 2 are all you need here.

In [ ]:
import torch

a = torch.tensor(7.0)
b = torch.tensor([1.0, 2.0, 3.0])
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

print(a.shape, b.shape, x.shape)
print(x.dtype, x.device)

**What each line does**

- `import torch`: loads PyTorch. Every PyTorch tool lives under the name `torch`.
- `torch.tensor(7.0)`: makes a 0-d tensor holding one number.
- `torch.tensor([1.0, 2.0, 3.0])`: makes a 1-d tensor from a Python list.
- `torch.tensor([[...], [...]])`: a list of lists makes a 2-d tensor: here, 2 rows and 3 columns.
- `.shape`: the tensor's size along each dimension. `torch.Size([2, 3])` means 2 rows × 3 columns; an empty `torch.Size([])` means a single number.
- `.dtype`: the kind of number stored. `float32` is a decimal number taking 4 bytes.
- `.device`: where the tensor lives: `cpu` (main processor) or, on a Mac, `mps` (the GPU, section 8).

Three words you just met, and will see constantly:

- **shape**: how many numbers along each dimension, e.g. `(2, 3)`.
- **dtype**: the number format, e.g. `float32` (4 bytes) or `bfloat16` (2 bytes).
- **device**: which chip holds the tensor, e.g. `cpu` or `mps`.

**Predict:** how many bytes does `x` take in memory? (6 numbers, 4 bytes each.)

In [ ]:
print(x.numel(), "numbers ×", x.element_size(), "bytes =", x.numel() * x.element_size(), "bytes")
print("in bfloat16:", x.to(torch.bfloat16).element_size() * x.numel(), "bytes")

**What each line does**

- `x.numel()`: counts the numbers in the tensor (2 × 3 = 6).
- `x.element_size()`: bytes per number for this dtype (4 for `float32`).
- `x.to(torch.bfloat16)`: makes a copy stored as `bfloat16`, a 2-byte format that big models use to save memory.
- Why this matters: an 8-billion-parameter model in `bfloat16` is about 8B × 2 bytes ≈ 16 GB. That arithmetic decides what fits on your Mac later in the course.

### Matrix multiply (`@`)

The most important operation in a neural network is **matrix multiply**, written `@` in PyTorch. It's how each building block of a network (a *layer*, explained in section 4) combines its inputs.

For `A @ B`, each result number is "one row of `A` times one column of `B`, added up". Here it is by hand, for a (2 × 3) times a (3 × 2):

```
A = [[1, 2, 3],        B = [[1, 0],
     [4, 5, 6]]             [0, 1],
                            [1, 1]]

row 1 of A · column 1 of B = 1×1 + 2×0 + 3×1 = 4
row 1 of A · column 2 of B = 1×0 + 2×1 + 3×1 = 5
row 2 of A · column 1 of B = 4×1 + 5×0 + 6×1 = 10
row 2 of A · column 2 of B = 4×0 + 5×1 + 6×1 = 11

A @ B = [[ 4,  5],
         [10, 11]]
```

The shape rule: **(a, b) @ (b, c) gives (a, c)**. The middle numbers must match (here 3 and 3), and they "disappear". Check it in code:

In [ ]:
A = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
B = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
print(A @ B)
print((A @ B).shape)

**What each line does**

- `A @ B`: matrix multiply. Each output number is a row of `A` times a column of `B`, summed.
- `(A @ B).shape`: (2, 3) @ (3, 2) gives (2, 2), exactly as the shape rule says.

### Broadcasting

When you add a tensor of shape `(3,)` to one of shape `(2, 3)`, PyTorch adds it to *every row*. This automatic stretching is called **broadcasting**. It's how one bias value per output gets added to every example in a group.

In [ ]:
bias = torch.tensor([10.0, 20.0, 30.0])  # shape (3,)
print(x + bias)  # (2, 3) + (3,) -> (2, 3)

**What each line does**

- `bias`: three numbers, one per column of `x`.
- `x + bias`: PyTorch adds `bias` to row 1 and again to row 2. You never write the loop yourself.
- `# ...`: anything after `#` is a comment, a note for humans that Python ignores.

**Predict:** what's the shape of `x @ torch.ones(3, 4)`? Use the shape rule, then run it.

In [ ]:
print((x @ torch.ones(3, 4)).shape)

**What each line does**

- `torch.ones(3, 4)`: a 3 × 4 tensor filled with 1s. Handy for trying out shapes.
- (2, 3) @ (3, 4) gives (2, 4): the matching 3s disappear.

## 2. Gradients and autograd

Back to step 3 of the training loop: *which way should each parameter move?*

Imagine the loss as a hilly landscape and the parameter as your position. The **gradient** tells you how steep the ground is where you're standing, and which way is uphill. To lower the loss, step the *opposite* way, downhill.

- A **negative** gradient means "increasing this parameter lowers the loss", so increase it.
- A **positive** gradient means the opposite, so decrease it.
- A gradient near **zero** means you're at (or near) the bottom.

Here's a toy loss with one parameter, `loss = (w − 5)²`. It's smallest at `w = 5`. Let's look at the slope at `w = 3`.

In [ ]:
ws = torch.linspace(0, 10, 100)
plt.figure(figsize=(5, 3.2))
plt.plot(ws, (ws - 5) ** 2, label="loss = (w - 5)²")
plt.plot([2, 4], [4 + 4, 4 - 4], "r-", lw=2, label="slope at w = 3 is -4")
plt.scatter([3], [4], color="red", zorder=3)
plt.xlabel("w")
plt.ylabel("loss")
plt.legend()
plt.show()

**What each line does**

- `torch.linspace(0, 10, 100)`: 100 evenly spaced numbers from 0 to 10, to draw a smooth curve.
- `(ws - 5) ** 2`: the loss for every one of those `w` values at once. `**` means 'to the power of'.
- The red line is the slope at `w = 3`. It goes *down* to the right, so the slope is negative (−4): moving `w` up lowers the loss.

Working out gradients by hand gets painful fast. **Autograd** is PyTorch's automatic gradient calculator. You mark a tensor with `requires_grad=True`, PyTorch remembers every calculation done with it, and `.backward()` works out the gradient for you.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
loss = (w - 5) ** 2
loss.backward()
print("gradient of the loss with respect to w:", w.grad)

**What each line does**

- `requires_grad=True`: tells PyTorch 'I'll want the gradient for this one; record what happens to it'.
- `loss = (w - 5) ** 2`: an ordinary calculation, but PyTorch is quietly recording it.
- `loss.backward()`: walks back through the recorded steps and works out the gradient of `loss` for every tensor marked `requires_grad`.
- `w.grad`: where the answer is stored: −4, matching the red slope above. (By hand: 2 × (w − 5) = 2 × (3 − 5) = −4.)

Now do the full loop by hand. The update rule is

&nbsp;&nbsp;&nbsp;&nbsp;`w ← w − learning_rate × gradient`

The **learning rate** is how big a step to take. Too small and training crawls; too big and you overshoot the bottom.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
lr = 0.1
for _ in range(25):
    loss = (w - 5) ** 2
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
    w.grad = None
print(f"w after 25 steps: {w.item():.4f}   (the best value is 5)")

**What each line does**

- `for _ in range(25):`: repeat the indented lines 25 times. `_` is the usual name for a loop counter you don't use.
- `loss = ...` and `loss.backward()`: steps 1–3 of the training loop: guess, measure, find the direction.
- `with torch.no_grad():`: switches off recording for the indented lines. The update itself is not part of the model's maths, so we don't want it recorded (PyTorch refuses to change a recorded tensor in place otherwise).
- `w -= lr * w.grad`: step 4, the nudge. `-=` means 'subtract from w and keep the result in w'.
- `w.grad = None`: throws the old gradient away before the next round. The next cell shows why this matters.
- `w.item()`: turns a one-number tensor into a plain Python number, for printing.

**Why reset the gradient?** Gradients *add up*: each `.backward()` adds its result to whatever is already in `.grad`. Watch what happens if you don't reset:

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
((w - 5) ** 2).backward()
print("after one backward():", w.grad)
((w - 5) ** 2).backward()
print("after two, no reset:  ", w.grad, " <- doubled, which is wrong for a fresh step")

**What each line does**

- The same calculation and `.backward()` run twice, with no `w.grad = None` in between.
- The second gradient is *added* to the first (−4 + −4 = −8). In a training loop that would mix old steps into new ones, so we always reset first.

**Predict:** go back two cells and change `lr = 0.1` to `lr = 1.1`. What happens to `w`? Run it and see, then set it back to `0.1`. (Hint: each step overshoots the bottom by more than it started.)

## 3. Linear regression, by hand

Now a real (tiny) model. We'll make 100 noisy points that roughly follow `y = 2x + 1`, then train `pred = x @ w + b` to find `w ≈ 2` and `b ≈ 1` on its own.

To measure how wrong the predictions are, we'll use the **mean squared error (MSE)**: for every point, take (prediction − truth), square it so it's always positive, then average. That average is the loss.

Exercises 1–3 ask you to write these same pieces as functions.

In [ ]:
torch.manual_seed(0)
X = torch.rand(100, 1) * 4
Y = 2 * X + 1 + 0.2 * torch.randn(100, 1)

w = torch.zeros(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

for step in range(500):
    pred = X @ w + b
    loss = ((pred - Y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        for p in (w, b):
            p -= 0.05 * p.grad
            p.grad = None
    if step % 100 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

print(f"learned w = {w.item():.3f}, b = {b.item():.3f}   (true: 2, 1)")

**What each line does**

- `torch.manual_seed(0)`: fixes the random numbers, so you get the same results every run.
- `torch.rand(100, 1) * 4`: 100 random inputs between 0 and 4, as a (100, 1) tensor: 100 rows, one column.
- `Y = 2 * X + 1 + 0.2 * torch.randn(100, 1)`: the true answers, plus a little random noise (`randn` gives bell-curve random numbers).
- `w`, `b`: the parameters, starting at zero, marked `requires_grad` so autograd tracks them.
- `pred = X @ w + b`: step 1, guess. (100, 1) @ (1, 1) gives (100, 1); `b` is broadcast to every row. This is exactly a *linear layer* (section 4 introduces it properly): exercise 1.
- `((pred - Y) ** 2).mean()`: step 2, measure: the MSE. `.mean()` averages all the numbers into one. This is exercise 2.
- `loss.backward()`: step 3, gradients for both `w` and `b`.
- The `torch.no_grad()` block: step 4, nudge each parameter and reset its gradient. This is exercise 3.
- `if step % 100 == 0`: print every 100th step (`%` is the remainder), so you can watch the loss fall.

## 4. nn.Module, layers and optimizers

Writing every parameter by hand doesn't scale to millions of them. PyTorch gives you ready-made pieces:

- A **layer** is a reusable chunk of a model. `nn.Linear(in, out)` is the layer you just wrote by hand (`x @ w + b`); it creates and stores its own `w` and `b`.
- **nn.Module** is the base for every model. Anything built from it keeps track of all its parameters for you.
- An **optimizer** does step 4 (the nudge) for every parameter at once. `torch.optim.SGD` is the plain "subtract learning rate × gradient" rule from section 2.

The loop then always has the same five beats:

1. forward: `pred = model(X)`
2. loss
3. `optimizer.zero_grad()`: reset old gradients
4. `loss.backward()`: compute new ones
5. `optimizer.step()`: nudge every parameter

In [ ]:
from torch import nn

model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for _ in range(500):
    pred = model(X)
    loss = ((pred - Y) ** 2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("weight", model.weight.item(), "bias", model.bias.item())
print("number of parameters:", sum(p.numel() for p in model.parameters()))

**What each line does**

- `from torch import nn`: `nn` (neural network) holds the ready-made layers.
- `nn.Linear(1, 1)`: a linear layer with 1 input and 1 output. It makes its own `weight` and `bias`, with `requires_grad` already on.
- `model.parameters()`: lists every parameter in the model, so the optimizer knows what to update.
- `torch.optim.SGD(..., lr=0.05)`: the optimizer. SGD stands for *stochastic gradient descent*: the update rule from section 2.
- `model(X)`: calling a model runs it on the input (its *forward* pass). Same result as `X @ w + b`.
- `optimizer.zero_grad()`: resets every parameter's gradient, like `p.grad = None` for each one.
- `optimizer.step()`: nudges every parameter, like the `torch.no_grad()` block you wrote in section 3.
- `sum(p.numel() for p in ...)`: counts all the numbers in all the parameters: 1 weight + 1 bias = 2.

## 5. Classification: logits, softmax and cross-entropy

So far the model predicted a *number* (a price, a `y`). Often you want it to pick a **class** instead: cat or dog, spam or not, or for a language model, *which word comes next*.

For that, the model outputs one score per class. These raw scores are called **logits**. They can be any number, positive or negative; bigger means "more likely".

- **softmax** turns logits into probabilities: all between 0 and 1, adding up to 1. (It raises *e* to each logit, then divides each by the total.)
- **argmax** picks the position of the biggest value, which is the predicted class. Softmax never changes which one is biggest, so you can take the argmax of the logits directly.
- **cross-entropy** is the loss for classification: minus the log of the probability the model gave to the *correct* class. If the model gave the right class probability 1, the loss is 0; the less probability it gave, the bigger the loss.

Here it is by hand, with three classes:

In [ ]:
import torch.nn.functional as F

logits = torch.tensor([2.0, 1.0, -1.0])  # scores for classes 0, 1, 2
probs = logits.exp() / logits.exp().sum()  # softmax, by hand
print("probabilities:", probs, "sum =", probs.sum().item())
print("predicted class:", logits.argmax().item())

correct = 0
print(f"cross-entropy by hand: {-torch.log(probs[correct]).item():.4f}")
pytorch_loss = F.cross_entropy(logits.unsqueeze(0), torch.tensor([correct]))
print(f"F.cross_entropy:       {pytorch_loss.item():.4f}")

**What each line does**

- `import torch.nn.functional as F`: `F` holds PyTorch's ready-made functions, including the loss functions.
- `logits`: three made-up scores. Class 0 has the biggest score.
- `logits.exp() / logits.exp().sum()`: softmax. `.exp()` is *e* to the power of each number; dividing by the total makes them add up to 1.
- `logits.argmax()`: the position of the biggest value: class 0.
- `-torch.log(probs[correct])`: cross-entropy by hand. `probs[0]` is the probability given to the correct class; `torch.log` is the natural logarithm.
- `F.cross_entropy(...)`: PyTorch's version, which does softmax and the log for you. It expects several examples at once (a *batch*, explained in section 6), as a tensor of shape (examples, classes), hence `.unsqueeze(0)` to turn (3,) into (1, 3), and one correct class per example.
- Pass **logits** (not probabilities) to `F.cross_entropy`. That's why your network in exercise 4 returns raw logits, with no softmax.

### Many examples at once: predictions and accuracy

Real models score many examples at once, one row each. To get every row's predicted class, take the argmax *along each row* with `dim=1`. Then compare with the correct classes using `==`, which gives True/False for each example.

In [ ]:
two_logits = torch.tensor(
    [
        [2.0, 1.0, -1.0],  # example 1: class 0 scores highest
        [0.1, 0.3, 2.5],
    ]
)  # example 2: class 2 scores highest
labels = torch.tensor([0, 1])  # the correct classes

preds = two_logits.argmax(dim=1)
print("predicted:", preds)
print("correct?  ", preds == labels)
print("fraction right:", (preds == labels).float().mean().item())

**What each line does**

- `two_logits`: shape (2, 3): two examples (rows), three class scores each (columns).
- `.argmax(dim=1)`: for each row, the position of its biggest value. `dim=1` means 'look across the columns'. Result: one predicted class per example.
- `preds == labels`: compares the two tensors position by position and gives `True` or `False` for each example. Here example 1 is right and example 2 is wrong.
- `.float()`: turns True/False into 1.0/0.0. You need this because you can't average True/False directly (PyTorch raises an error).
- `.mean()`: the average of the 1s and 0s is the fraction you got right: 0.5 here.

**Predict:** change `correct = 0` to `correct = 2` (the class with the lowest score) and rerun. Will the loss go up or down? Why?

### One more piece: ReLU

A stack of linear layers is still just one linear layer in disguise. It can only draw straight-line boundaries between classes. To bend the boundary, put a simple non-linear function between layers. The most common is **ReLU**: keep positive numbers, turn negatives into 0.

In [ ]:
z = torch.tensor([-2.0, -0.5, 0.0, 1.5, 3.0])
print(torch.relu(z))

**What each line does**

- `torch.relu(z)`: replaces every negative number with 0 and leaves the rest alone.
- That's the whole of ReLU. Yet putting it between two linear layers is what lets a network learn curved boundaries, which you'll need for the two moons.

## 6. Batches and epochs

With lots of data, you don't feed every example through the model at once for each step. Instead:

- A **batch** (or *mini-batch*) is a small group of examples, say 64, used for one training step. Each batch gives one loss, one `.backward()` and one `.step()`.
- An **epoch** is one full pass through all the training data, batch by batch.

Shuffling the order each epoch stops the model from learning anything from the order of the examples. Here's what "shuffle, then slice into batches" looks like for 10 examples and a batch size of 4:

In [ ]:
n, batch_size = 10, 4
order = torch.randperm(n)
print("shuffled order:", order.tolist())
for start in range(0, n, batch_size):
    idx = order[start : start + batch_size]
    print("  batch:", idx.tolist())

**What each line does**

- `torch.randperm(n)`: the numbers 0 to n−1 in random order: a shuffled list of row positions.
- `range(0, n, batch_size)`: 0, 4, 8: where each batch starts.
- `order[start : start + batch_size]`: a *slice*: the positions from `start` up to (not including) `start + batch_size`. The last batch is smaller when the sizes don't divide evenly.
- With a data tensor `X`, `X[idx]` would pick out exactly those rows. `.tolist()` only turns tensors into plain lists for printing.

## 7. A worked example: a small network, end to end

Here everything comes together in a tiny network that learns the curve `y = sin(x)`. Read it slowly. Exercises 4 and 5 follow the **same pattern**: a class built from `nn.Module` with `__init__` and `forward`, and a loop with the five beats.

It differs from the exercises on purpose: it predicts a number (so MSE loss, not cross-entropy), uses `Tanh` instead of ReLU, and trains on all the data in every step rather than in batches.

**A note on classes.** In Python, `class SineNet(nn.Module):` defines a new kind of model, built on `nn.Module`.

- `__init__` runs once when you create the model. It's where you create the layers and store them on `self`, which means "this model".
- `super().__init__()` must come first: it sets up the `nn.Module` machinery that tracks parameters.
- `forward` says what happens to an input `x`, step by step. Calling `model(x)` runs `forward`.

In [ ]:
torch.manual_seed(0)


class SineNet(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()
        self.layer_in = nn.Linear(1, hidden)
        self.layer_out = nn.Linear(hidden, 1)

    def forward(self, x):
        h = torch.tanh(self.layer_in(x))
        return self.layer_out(h)


xs = torch.linspace(-3, 3, 200).unsqueeze(1)
ys = torch.sin(xs)

net = SineNet(hidden=32)
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)

for step in range(1000):
    pred = net(xs)
    loss = ((pred - ys) ** 2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 250 == 0:
        print(f"step {step:4d}  loss {loss.item():.4f}")

with torch.no_grad():
    plt.figure(figsize=(5, 3.2))
    plt.plot(xs, ys, label="sin(x)")
    plt.plot(xs, net(xs), "--", label="the network")
    plt.legend()
    plt.show()

**What each line does**

- `class SineNet(nn.Module):`: a new model, built on `nn.Module` so it tracks its own parameters.
- `super().__init__()`: sets up `nn.Module`. Always the first line of `__init__`.
- `self.layer_in = nn.Linear(1, hidden)`: one input in, `hidden` numbers out. Storing it on `self` registers its parameters with the model.
- `self.layer_out = nn.Linear(hidden, 1)`: `hidden` numbers in, one prediction out. The shapes chain: 1 → hidden → 1.
- `forward`: first layer, then `torch.tanh` (a curvy non-linear function, like ReLU's job), then the second layer. The result is returned.
- `torch.linspace(-3, 3, 200).unsqueeze(1)`: 200 inputs as a (200, 1) tensor; `unsqueeze(1)` adds the 'one column' dimension that `nn.Linear` expects.
- `torch.optim.Adam`: a smarter optimizer than SGD that adapts the step size for each parameter. Fine-tuning almost always uses Adam (or its cousin AdamW).
- The loop: the same five beats as section 4: forward, loss, `zero_grad()`, `backward()`, `step()`.
- `with torch.no_grad():` around the plot: we're only looking at predictions, not training, so there's no need to record anything.

Compare this with what exercises 4 and 5 ask for:

| | Worked example | Your exercises |
|---|---|---|
| Model | `SineNet`: Linear → **tanh** → Linear | `MLP`: Linear → **ReLU** → Linear |
| Output | one number | one **logit per class** |
| Loss | MSE | **cross-entropy** (`F.cross_entropy`) |
| Data per step | all of it | shuffled **batches** (section 6) |
| One call of the training code does | 1000 steps | **one epoch**, then returns the average loss |

## 8. The Mac's GPU: MPS

Your Mac has a GPU (graphics processor) that's far faster than the CPU at the big matrix multiplies networks are made of. PyTorch reaches it through a device called **MPS** (Metal Performance Shaders).

One rule: a model and the data you feed it must be on the **same device**. Move them with `.to(device)`.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("using", device)

big = torch.randn(4096, 4096, device=device)
out = big @ big
print(out.shape, out.device)

small_net = SineNet(hidden=8).to(device)
print(small_net(torch.zeros(3, 1, device=device)).device)

**What each line does**

- `torch.backends.mps.is_available()`: True on an Apple Silicon Mac with a recent PyTorch. Otherwise we fall back to the CPU.
- `torch.randn(4096, 4096, device=device)`: makes a big random tensor directly on the GPU.
- `big @ big`: about 69 billion multiply-adds (4096 × 4096 × 4096), done in a blink on MPS.
- `.to(device)`: moves a model (all its parameters) or a tensor to that device. The input here was also made on `device`, so they match.

Your Mac's memory is *unified*: the GPU uses the same RAM as everything else. Keep an eye on Activity Monitor's memory pressure graph as models get bigger later in the course.

## 9. Your task: classify two moons

Below are two classes of points shaped like interlocking half-moons. No straight line can separate them, so you'll build a network with a hidden layer and a ReLU between two linear layers (section 5), and train it with batches (section 6) the way section 7 trained `SineNet`.

In [ ]:
import sys

sys.path.insert(0, ".")  # so Python can find data.py next to this notebook
from data import make_moons

Xm, ym = make_moons(n=1000, noise=0.2)
plt.figure(figsize=(5, 3.5))
plt.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap="coolwarm", s=8)
plt.title("Two moons: can a network separate them?")
plt.show()

**What each line does**

- `sys.path.insert(0, ".")`: tells Python to also look in this notebook's folder for files to import, so it finds `data.py`.
- `make_moons(n=1000, noise=0.2)`: makes 1000 points: `Xm` has shape (1000, 2), an (x, y) position per point, and `ym` holds each point's class, 0 or 1.
- `Xm[:, 0]`: picks column 0 of every row (all the x positions). The `:` means 'every row'. `Xm[:, 1]` is column 1, the y positions.
- `plt.scatter(..., c=ym, cmap="coolwarm", s=8)`: draws a dot per point, coloured by its class (`c`), using a blue-to-red colour scheme (`cmap`), at size 8 (`s`).

### What to do now

1. Open `exercises.py`. Each exercise says in plain words what it is, and has numbered steps to fill in. Exercises 4 and 5 follow section 7's pattern.
2. Check your work as you go: `make check M=01`. (The training test keeps failing until you run train.py in step 4 below.)
3. Answer the six questions in `answers.toml`. You can work them out in this notebook.
4. Train your network: `uv run python modules/m01_pytorch_essentials/train.py`
5. Run `make check M=01` again. When everything passes, M1 turns green on the dashboard.

Stuck? `make hint M=01 EX=3` gives a nudge for exercise 3, and every term is in the glossary in `README.md`.

**Stretch:** re-run `train.py --hidden 4`, then `--hidden 256`, tagging each check with `make check M=01 LABEL="h=4"`. Compare the attempts on the dashboard.